# Gink's own model: a first fine-tune (free Colab)

This notebook teaches an open model (**Qwen3.5-4B**, Apache-2.0) Gink's actions, tone and languages from the
Gink dataset, then scores it on examples it never saw. It's the **pilot**: it proves the pipeline works before
anything costs money. See `tools/gink_model/PLAN.md` for the whole plan.

**Before you start:** Runtime → Change runtime type → **T4 GPU** (free). It takes about 30–60 minutes.

Rules it follows: only Gink's own templates and hand-written examples (no users' chats, no Claude output);
the base model's licence notice is kept.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Get the project
Either clone it from GitHub (if the repository is public), or upload `GinkAssistant_full.zip` when asked.

In [ ]:
import os, subprocess
if not os.path.exists("GinkAssistant"):
    ok = subprocess.run(["git", "clone", "--depth", "1", "https://github.com/Amgib00/GinkAssistant"]).returncode == 0
    if not ok:
        from google.colab import files
        print("Couldn't clone (a private repository?). Upload GinkAssistant_full.zip:")
        up = files.upload()
        name = next(iter(up))
        subprocess.run(["unzip", "-q", "-o", name, "-d", "GinkAssistant"], check=True)
%cd GinkAssistant

In [ ]:
!pip install -q unsloth

## 2. Build and check the dataset
`validate.py` must say **0 with problems**.

In [ ]:
!python tools/gink_dataset/generate.py && python tools/gink_dataset/validate.py

## 3. Train
LoRA on Qwen3.5-4B in bf16 (Unsloth advises against 4-bit QLoRA for Qwen3.5). At the end it prints a
**held-out score** -- write it down. `--gguf q4_k_m` also makes a file the desktop app's offline AI
(llama.cpp) can load.

In [ ]:
!python tools/gink_model/train_unsloth.py --base unsloth/Qwen3.5-4B --out gink-4b --epochs 2 --gguf q4_k_m

## 4. Download the result
The LoRA adapter is small (tens of MB). The GGUF is a few GB: download it only if you want to try it in
llama.cpp / the desktop app's offline mode.

In [ ]:
!zip -q -r gink-4b-lora.zip gink-4b/lora gink-4b/GINK_MODEL.json
from google.colab import files
files.download("gink-4b-lora.zip")

## 5. Next
1. Compare with today's Gink (`gink-crystal`) on the same examples:
   `python tools/gink_dataset/evaluate.py --base-url https://gink-server.onrender.com/v1 --model gink-crystal --save baseline.json`
2. If the pilot looks promising, grow the dataset (PLAN.md, step 2) and train the 9B model on a hosted
   service (PLAN.md, step 3).